In [11]:
import pandas as pd

df_election = pd.read_csv('src/election.csv', sep=';')
df_population = pd.read_csv('src/population.csv', sep=';', dtype={'GEO': str})
df_conjugality = pd.read_csv('src/conjugality.csv', sep=';', dtype={'GEO': str})
df_household = pd.read_csv('src/household.csv', sep=';', dtype={'GEO': str})


def convert_values_to_rates(df):
    columns = [col for col in df.columns if col != 'POPULATION_TOTAL']
    df = df[columns].div(df['POPULATION_TOTAL'], axis=0) * 100
    return df

df_cities = df_population[['GEO', 'CITY']]
df_cities = df_cities.drop_duplicates()
df_cities = df_cities.drop_duplicates(subset=['CITY']).set_index('CITY')
df_cities = df_cities.rename(columns={'GEO': 'CODE_INSEE'})

### DF conjugality

In [12]:
display(df_conjugality.sort_values(by='CITY'))


df_conjugality_civil_status = df_conjugality[
    (df_conjugality['AGE'] == '15 ans ou plus') & (df_conjugality['COUPLE'] == 'Total')
].pivot_table(
    index='CITY',
    columns='CIVIL_STATUS',
    values='POPULATION',
    aggfunc='sum',
    fill_value=0
).rename(columns={
    'Marié': 'MARRIED', 'Pacsé': 'CIVIL_PARTNER', 'En concubinage, union libre': 'LIVEIN_PARTNER', 'Divorcé': 'DIVORCED', 'Célibataire': 'SINGLE', 'Veuf': 'WIDOWED', 'Total': 'POPULATION_TOTAL'
})


df_conjugality_age = df_conjugality[
    (df_conjugality['COUPLE'] == 'Total') & (df_conjugality['CIVIL_STATUS'] == 'Total')
].pivot_table(
    index='CITY',
    columns='AGE',
    values='POPULATION',
    aggfunc='sum',
    fill_value=0
).rename(columns={
    'De 15 à 24 ans ': 'AGE_15_24', 'De 25 à 39 ans': 'AGE_25_39', 'De 40 à 54 ans': 'AGE_40_54', 'De 55 à 64 ans': 'AGE_55_64', 'De 65 à 79 ans': 'AGE_65_79', '80 ans ou plus': 'AGE_80+', '15 ans ou plus': 'POPULATION_TOTAL'
})

df_conjugality_final = pd.concat([df_conjugality_age, df_conjugality_civil_status], axis=1)
df_conjugality_final = df_conjugality_final.drop('POPULATION_TOTAL', axis=1)
display(df_conjugality_final)
df_conjugality_final.columns

,AGE,CIVIL_STATUS,COUPLE,GEO,POPULATION,CITY
557783,De 40 à 54 ans,Total,Oui,64001,32.980769,Aast
605600,De 15 à 24 ans,Total,Oui,64001,0.942308,Aast
772040,De 40 à 54 ans,Total,Total,64001,40.519231,Aast
778506,De 15 à 24 ans,Total,Total,64001,15.076923,Aast
551262,De 55 à 64 ans,Total,Non,64001,5.653846,Aast
...,...,...,...,...,...,...
725413,De 65 à 79 ans,Total,Total,51410,118.815261,Œuilly
467339,De 65 à 79 ans,Total,Non,51410,38.150480,Œuilly
375528,De 40 à 54 ans,Total,Non,02565,8.000000,Œuilly
137694,15 ans ou plus,Pacsé,Total,51410,33.193869,Œuilly


,AGE_80+,AGE_15_24,AGE_25_39,AGE_40_54,AGE_55_64,AGE_65_79,SINGLE,DIVORCED,LIVEIN_PARTNER,MARRIED,CIVIL_PARTNER,WIDOWED
CITY,,,,,,,,,,,,
Aast,8.480769,15.076923,31.096154,40.519231,33.923077,38.634615,23.557692,10.365385,19.788462,80.096154,27.326923,6.596154
Abainville,10.518373,24.251077,38.544087,60.194238,51.482521,58.131857,44.751015,12.648965,15.720836,142.392433,12.788927,14.819979
Abancourt,45.536966,108.367303,178.520102,192.779920,138.701520,193.858726,171.132294,41.587759,126.655357,396.935151,60.187553,61.266422
Abaucourt,16.000000,19.000000,64.000000,57.000000,45.000000,56.000000,49.000000,12.000000,37.000000,122.000000,22.000000,15.000000
Abaucourt-Hautecourt,2.941176,11.764706,22.549020,20.588235,14.705882,18.627451,27.450980,3.921569,18.627451,30.392157,5.882353,4.901961
...,...,...,...,...,...,...,...,...,...,...,...,...
Île-de-Sein,27.084316,10.949197,25.590268,46.086140,67.530371,87.611033,58.264900,25.992994,14.409246,126.410396,10.741537,29.032253
Ô-de-Selle,58.636275,133.190095,173.898089,238.510681,155.554580,201.278353,223.069229,42.401174,115.126894,399.529036,113.324382,67.617358
Œting,126.524521,210.960276,390.247557,588.167428,477.893465,403.503019,397.435662,97.423876,142.116353,1323.631065,111.572778,125.116531


Index(['AGE_80+', 'AGE_15_24', 'AGE_25_39', 'AGE_40_54', 'AGE_55_64',
       'AGE_65_79', 'SINGLE', 'DIVORCED', 'LIVEIN_PARTNER', 'MARRIED',
       'CIVIL_PARTNER', 'WIDOWED'],
      dtype='str')

### DF population

In [13]:
display(df_population.sort_values(by='CITY'))


df_population_sex = df_population[
    (df_population['AGE'] == '15 ans ou plus') & (df_population['SOCIO_PROF_CAT'] == 'Total')
].pivot_table(
    index='CITY',
    columns='SEX',
    values='POPULATION',
    aggfunc='sum',
    fill_value=0
).rename(columns={'Femme': 'SEX_WOMEN', 'Homme': 'SEX_MEN', 'Total': 'POPULATION_TOTAL'})


df_population_spc = df_population[
    (df_population['AGE'] == '15 ans ou plus') & (df_population['SEX'] == 'Total')
].pivot_table(
    index='CITY',
    columns='SOCIO_PROF_CAT',
    values='POPULATION',
    aggfunc='sum',
    fill_value=0
).rename(columns={
    'Agriculteurs': 'SPC_FARMERS',
    'Artisans, commerçants et chefs d’entreprise': 'SPC_BUSINESS',
    'Cadres et professions intellectuelles supérieures': 'SPC_EXECUTIVES',
    'Professions intermédiaires': 'SPC_INTERMEDIATES',
    'Employés': 'SPC_EMPLOYEES',
    'Ouvriers': 'SPC_WORKERS',
    'Autres inactifs': 'SPC_INACTIVES',
    'Retraités': 'SPC_RETIRED',
    'Etudiants ou élèves': 'SPC_STUDENTS',
    'Total': 'POPULATION_TOTAL'
})

display(df_population_spc)

df_population_final = pd.concat([df_population_sex, df_population_spc], axis=1)
df_population_final = df_population_final.drop('POPULATION_TOTAL', axis=1)
display(df_population_final)
df_population_final.columns


,AGE,GEO,SOCIO_PROF_CAT,SEX,POPULATION,CITY
2101362,15 ans ou plus,64001,Total,Total,157.756098,Aast
1915817,15 ans ou plus,64001,Retraités,Homme,33.463415,Aast
1912216,15 ans ou plus,64001,Ouvriers,Homme,19.121951,Aast
739300,De 25 à 54 ans,64001,"Artisans, commerçants et chefs d’entreprise",Total,4.780488,Aast
629700,De 25 à 54 ans,64001,Professions intermédiaires,Total,28.682927,Aast
...,...,...,...,...,...,...
1729758,15 ans ou plus,51410,"Artisans, commerçants et chefs d’entreprise",Total,21.631020,Œuilly
1596115,De 15 à 24 ans,02565,Total,Femme,14.274194,Œuilly
1595837,De 15 à 24 ans,02565,Total,Total,42.822581,Œuilly
691725,55 ans ou plus,51410,Retraités,Homme,60.542525,Œuilly


SOCIO_PROF_CAT,SPC_FARMERS,SPC_BUSINESS,SPC_INACTIVES,SPC_EXECUTIVES,SPC_EMPLOYEES,SPC_WORKERS,SPC_INTERMEDIATES,SPC_RETIRED,POPULATION_TOTAL
CITY,,,,,,,,,
Aast,0.000000,4.780488,9.560976,0.000000,23.902439,19.121951,33.463415,66.926829,157.756098
Abainville,0.000000,0.000000,52.017327,14.617605,38.532471,32.844527,23.914866,83.986959,245.913755
Abancourt,5.650101,33.179218,131.178015,43.825932,86.826549,177.375831,76.781892,322.535795,877.353333
Abaucourt,10.218750,0.000000,10.218750,15.328125,35.765625,30.656250,35.765625,132.843750,270.796875
Abaucourt-Hautecourt,11.111111,5.555556,22.222222,0.000000,11.111111,22.222222,5.555556,16.666667,94.444444
...,...,...,...,...,...,...,...,...,...
Île-de-Sein,0.000000,37.585845,15.330019,16.560193,39.005460,9.485478,26.870194,117.868214,262.705402
Ô-de-Selle,20.372731,25.379768,85.629503,128.530871,99.799142,118.148727,201.362307,291.682959,970.906008
Œting,0.000000,70.114677,345.235419,295.676329,335.689870,277.696357,389.310521,490.105275,2203.828448


,SEX_WOMEN,SEX_MEN,SPC_FARMERS,SPC_BUSINESS,SPC_INACTIVES,SPC_EXECUTIVES,SPC_EMPLOYEES,SPC_WORKERS,SPC_INTERMEDIATES,SPC_RETIRED
CITY,,,,,,,,,,
Aast,76.487805,81.268293,0.000000,4.780488,9.560976,0.000000,23.902439,19.121951,33.463415,66.926829
Abainville,120.573948,125.339807,0.000000,0.000000,52.017327,14.617605,38.532471,32.844527,23.914866,83.986959
Abancourt,450.800237,426.553096,5.650101,33.179218,131.178015,43.825932,86.826549,177.375831,76.781892,322.535795
Abaucourt,132.843750,137.953125,10.218750,0.000000,10.218750,15.328125,35.765625,30.656250,35.765625,132.843750
Abaucourt-Hautecourt,50.000000,44.444444,11.111111,5.555556,22.222222,0.000000,11.111111,22.222222,5.555556,16.666667
...,...,...,...,...,...,...,...,...,...,...
Île-de-Sein,142.575335,120.130068,0.000000,37.585845,15.330019,16.560193,39.005460,9.485478,26.870194,117.868214
Ô-de-Selle,481.209144,489.696864,20.372731,25.379768,85.629503,128.530871,99.799142,118.148727,201.362307,291.682959
Œting,1110.209250,1093.619198,0.000000,70.114677,345.235419,295.676329,335.689870,277.696357,389.310521,490.105275


Index(['SEX_WOMEN', 'SEX_MEN', 'SPC_FARMERS', 'SPC_BUSINESS', 'SPC_INACTIVES',
       'SPC_EXECUTIVES', 'SPC_EMPLOYEES', 'SPC_WORKERS', 'SPC_INTERMEDIATES',
       'SPC_RETIRED'],
      dtype='str')

### DF household

In [14]:
df_household = df_household.drop(['PCS'], axis=1)

display(df_household.sort_values(by='CITY'))


df_household_final = df_household.pivot_table(
    index='CITY',
    columns='TPH',
    values='OBS_VALUE',
    aggfunc='sum',
    fill_value=0
).rename(columns={
    'Ménage à une personne': 'ONE_PERSON',
    'Homme seul': 'ONE_MAN',
    'Femme seule': 'ONE_WOMAN',
    'Ménage sans famille à plusieurs personnes ': 'HOUSESHARE',
    'Ménage comprenant une famille': 'FAMILY',
    'Famille principale couple sans enfants': 'FAMILY_CHILDLESS',
    'Famille principale monoparentale': 'FAMILY_SINGLE_PARENT',
    'Famille principale couple avec enfants': 'FAMILY_CHILDREN',
    'Total': 'POPULATION_TOTAL'
})

display(df_household_final)
df_household_final.columns

,GEO,TPH,OBS_VALUE,CITY
41902,64001,Ménage sans famille à plusieurs personnes,9.560976,Aast
39522,64001,Ménage à une personne,14.341463,Aast
39594,64001,Famille principale couple avec enfants,100.390244,Aast
37163,64001,Femme seule,4.780488,Aast
41471,64001,Famille principale couple sans enfants,62.146341,Aast
...,...,...,...,...
251694,51410,Homme seul,22.414481,Œuilly
244971,51410,Famille principale couple avec enfants,208.145060,Œuilly
248949,51410,Total,512.991104,Œuilly
248350,51410,Famille principale couple sans enfants,181.069300,Œuilly


TPH,FAMILY_CHILDREN,FAMILY_CHILDLESS,FAMILY_SINGLE_PARENT,ONE_WOMAN,ONE_MAN,FAMILY,HOUSESHARE,ONE_PERSON,POPULATION_TOTAL
CITY,,,,,,,,,
Aast,100.390244,62.146341,9.560976,4.780488,9.560976,172.097561,9.560976,14.341463,196.000000
Abainville,133.441056,108.627426,14.227153,10.281746,15.422619,256.295635,0.000000,25.704365,282.000000
Abancourt,463.061081,352.158341,83.322033,80.190665,42.065354,898.541455,10.202526,122.256019,1031.000000
Abaucourt,163.500000,102.187500,15.328125,25.546875,10.218750,281.015625,10.218750,35.765625,327.000000
Abaucourt-Hautecourt,33.333333,44.444444,0.000000,11.111111,11.111111,77.777778,0.000000,22.222222,100.000000
...,...,...,...,...,...,...,...,...,...
Île-de-Sein,50.108113,107.156517,0.000000,70.134497,52.600873,157.264630,0.000000,122.735370,280.000000
Ô-de-Selle,576.458339,284.721041,151.029672,45.275947,86.024299,1012.209053,9.490701,131.300246,1153.000000
Œting,1371.145943,772.534575,152.532035,186.044235,114.488760,2296.212553,39.254452,300.532994,2636.000000


Index(['FAMILY_CHILDREN', 'FAMILY_CHILDLESS', 'FAMILY_SINGLE_PARENT',
       'ONE_WOMAN', 'ONE_MAN', 'FAMILY', 'HOUSESHARE', 'ONE_PERSON',
       'POPULATION_TOTAL'],
      dtype='str', name='TPH')

## Merge des différents dataframes

### Merge des dataframes INSEE

In [15]:
df_insee = pd.concat([df_cities, df_population_final, df_household_final, df_conjugality_final], axis=1, join='inner')

df_insee = df_insee.reset_index()
df_insee = df_insee.sort_values('CODE_INSEE')

display(df_insee)
df_insee.columns

,CITY,CODE_INSEE,SEX_WOMEN,SEX_MEN,SPC_FARMERS,SPC_BUSINESS,SPC_INACTIVES,SPC_EXECUTIVES,SPC_EMPLOYEES,SPC_WORKERS,...,AGE_25_39,AGE_40_54,AGE_55_64,AGE_65_79,SINGLE,DIVORCED,LIVEIN_PARTNER,MARRIED,CIVIL_PARTNER,WIDOWED
11076,Beauregard,01030,469.003121,400.563806,4.678950,49.795161,59.954849,119.966302,138.692259,97.279474,...,191.648144,225.887517,149.881881,160.809887,206.416127,79.807165,116.072258,386.201493,51.406531,45.670426
20081,Belley,01034,4018.213916,3788.407131,14.951696,229.275446,1191.328925,368.808511,1201.554342,1325.717079,...,1441.000000,1648.000000,1200.000000,1552.000000,2066.000000,681.000000,754.000000,3132.000000,354.000000,754.000000
425,Bourg-en-Bresse,01053,18622.885204,16591.607556,10.565152,737.666302,7291.596104,2529.626729,5131.046097,4951.669281,...,7373.150111,7529.820169,4654.501820,6226.958453,12317.138236,3521.349490,3299.690055,12384.273852,1129.088969,2587.726049
407,Brens,01061,1520.053285,1398.315906,10.008457,159.993865,352.069703,270.866484,424.383192,290.139384,...,493.743661,819.447556,495.052868,554.442108,607.395017,150.523002,329.133611,1407.820753,241.162592,162.009046
347,Cerdon,01068,736.524307,681.910254,9.974085,64.584895,157.673928,74.133837,192.932476,178.279048,...,225.393957,324.975205,216.843657,325.743389,352.879219,87.694378,169.151540,594.608417,73.177016,150.973706
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4486,Sainte-Suzanne,97420,10706.794728,9604.020209,136.372196,575.876201,5748.058891,1086.327378,4276.527312,2785.942548,...,4705.813442,5604.854189,3473.034916,2513.739587,8142.410778,885.189927,2811.406450,7372.551603,501.690026,946.313936
464,Salazie,97421,2740.307107,2836.925200,207.887935,148.779093,1799.244479,61.700211,943.213397,1175.327670,...,1295.297747,1567.332570,920.135698,711.673239,2021.318596,182.186255,885.747391,2135.625612,42.077795,364.591721
5262,Le Tampon,97422,33811.692991,30513.284058,1180.842480,2872.608457,16695.430022,2590.061227,13002.171632,8290.660632,...,14640.915939,15820.512974,10702.949499,8832.705369,23050.868985,4443.439985,8225.022218,23842.224861,1470.169696,3182.951611
5820,Les Trois-Bassins,97423,2948.329422,2744.765843,55.003507,153.178643,1397.510115,208.397826,1350.268634,1051.467021,...,1256.730718,1626.307029,981.866515,790.522607,2293.304136,226.625105,796.401298,1987.170562,109.654431,341.745004


Index(['CITY', 'CODE_INSEE', 'SEX_WOMEN', 'SEX_MEN', 'SPC_FARMERS',
       'SPC_BUSINESS', 'SPC_INACTIVES', 'SPC_EXECUTIVES', 'SPC_EMPLOYEES',
       'SPC_WORKERS', 'SPC_INTERMEDIATES', 'SPC_RETIRED', 'FAMILY_CHILDREN',
       'FAMILY_CHILDLESS', 'FAMILY_SINGLE_PARENT', 'ONE_WOMAN', 'ONE_MAN',
       'FAMILY', 'HOUSESHARE', 'ONE_PERSON', 'POPULATION_TOTAL', 'AGE_80+',
       'AGE_15_24', 'AGE_25_39', 'AGE_40_54', 'AGE_55_64', 'AGE_65_79',
       'SINGLE', 'DIVORCED', 'LIVEIN_PARTNER', 'MARRIED', 'CIVIL_PARTNER',
       'WIDOWED'],
      dtype='str')

## Merge avec dataframe elections

### Dataframe complet pour l'analyse

In [16]:
# Merge initial
df_election = df_election.rename(columns={'Code_INSEE': 'CODE_INSEE'})
display(df_election)

df_merged = df_insee.merge(df_election, on='CODE_INSEE', how='left')

# Récupération des communes et population totale en début de tableau
col_population = df_merged.pop('POPULATION_TOTAL')
col_city_election = df_merged.pop('Libellé de la commune')
df_merged.insert(1, 'CITY_ELECTION', col_city_election)
df_merged.insert(3, 'POPULATION_TOTAL', col_population)


# Nettoyage des valeurs vides et de la colonne city en doublon
df_merged = df_merged.dropna()
df_merged = df_merged.drop('CITY_ELECTION', axis=1)

display(df_merged)


,CODE_INSEE,Libellé de la commune,Inscrits,Abstentions,Votants,Blancs,Nuls,Exprimés,pct_abstention,pct_blancs,pct_nuls,pct_gauche,pct_centre,pct_droite
0,01001,L'Abergement-Clémenciat,645,108,537,16,1,520,16.74,2.98,0.19,21.73,32.31,45.96
1,01002,L'Abergement-de-Varey,213,38,175,3,1,171,17.84,1.71,0.57,38.60,35.09,26.32
2,01004,Ambérieu-en-Bugey,8765,2078,6687,88,46,6553,23.71,1.32,0.69,34.44,24.87,40.68
3,01005,Ambérieux-en-Dombes,1282,234,1048,14,6,1028,18.25,1.34,0.57,21.50,29.47,49.03
4,01006,Ambléon,103,23,80,3,0,77,22.33,3.75,0.00,32.47,29.87,37.66
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35224,ZZ229,Zurich,24868,14101,10767,40,31,10696,56.70,0.37,0.29,26.67,54.92,18.41
35225,ZZ231,Taipei,1709,942,767,8,2,757,55.12,1.04,0.26,37.38,39.23,23.38
35226,ZZ233,Nour-Soultan,117,64,53,2,0,51,54.70,3.77,0.00,27.45,50.98,21.57
35227,ZZ234,Monterrey,713,553,160,0,2,158,77.56,0.00,1.25,20.89,61.39,17.72


,CITY,CODE_INSEE,POPULATION_TOTAL,SEX_WOMEN,SEX_MEN,SPC_FARMERS,SPC_BUSINESS,SPC_INACTIVES,SPC_EXECUTIVES,SPC_EMPLOYEES,...,Votants,Blancs,Nuls,Exprimés,pct_abstention,pct_blancs,pct_nuls,pct_gauche,pct_centre,pct_droite
0,Beauregard,01030,228.000000,469.003121,400.563806,4.678950,49.795161,59.954849,119.966302,138.692259,...,433.0,2.0,4.0,427.0,23.77,0.46,0.92,26.93,26.46,46.60
1,Belley,01034,208.000000,4018.213916,3788.407131,14.951696,229.275446,1191.328925,368.808511,1201.554342,...,4341.0,72.0,28.0,4241.0,26.89,1.66,0.65,31.86,27.89,40.25
2,Bourg-en-Bresse,01053,240.000000,18622.885204,16591.607556,10.565152,737.666302,7291.596104,2529.626729,5131.046097,...,16316.0,252.0,72.0,15992.0,29.54,1.54,0.44,37.24,30.30,32.45
3,Brens,01061,2367.000000,1520.053285,1398.315906,10.008457,159.993865,352.069703,270.866484,424.383192,...,777.0,13.0,5.0,759.0,18.30,1.67,0.64,26.09,28.06,45.85
4,Cerdon,01068,884.996576,736.524307,681.910254,9.974085,64.584895,157.673928,74.133837,192.932476,...,431.0,7.0,3.0,421.0,23.04,1.62,0.70,21.38,20.43,58.19
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29782,Villers-en-Arthies,95676,487.000000,210.196629,169.755939,0.000000,20.221523,39.248669,35.135483,79.137430,...,340.0,2.0,0.0,338.0,12.82,0.59,0.00,26.63,27.51,45.86
29783,Villiers-Adam,95678,848.000000,319.247059,359.152941,0.000000,54.870588,89.788235,114.729412,74.823529,...,531.0,9.0,5.0,517.0,11.94,1.69,0.94,25.73,31.33,42.94
29784,Villiers-le-Bel,95680,28710.329186,11144.103286,10452.072763,0.000000,639.813936,6013.240784,882.423154,4525.300632,...,8388.0,106.0,76.0,8206.0,32.36,1.26,0.91,59.10,19.34,21.56
29785,Villiers-le-Sec,95682,981.006831,391.217842,372.466984,4.400683,25.131365,106.179115,38.719162,164.221495,...,96.0,0.0,1.0,95.0,12.73,0.00,1.04,31.58,29.47,38.95


In [17]:
df_merged.to_csv('statistiques.csv', index=False, sep=';', encoding='utf-8')

In [18]:
df_merged.columns

Index(['CITY', 'CODE_INSEE', 'POPULATION_TOTAL', 'SEX_WOMEN', 'SEX_MEN',
       'SPC_FARMERS', 'SPC_BUSINESS', 'SPC_INACTIVES', 'SPC_EXECUTIVES',
       'SPC_EMPLOYEES', 'SPC_WORKERS', 'SPC_INTERMEDIATES', 'SPC_RETIRED',
       'FAMILY_CHILDREN', 'FAMILY_CHILDLESS', 'FAMILY_SINGLE_PARENT',
       'ONE_WOMAN', 'ONE_MAN', 'FAMILY', 'HOUSESHARE', 'ONE_PERSON', 'AGE_80+',
       'AGE_15_24', 'AGE_25_39', 'AGE_40_54', 'AGE_55_64', 'AGE_65_79',
       'SINGLE', 'DIVORCED', 'LIVEIN_PARTNER', 'MARRIED', 'CIVIL_PARTNER',
       'WIDOWED', 'Inscrits', 'Abstentions', 'Votants', 'Blancs', 'Nuls',
       'Exprimés', 'pct_abstention', 'pct_blancs', 'pct_nuls', 'pct_gauche',
       'pct_centre', 'pct_droite'],
      dtype='str')

In [21]:
import pandas as pd
import json

# --- 1. On définit la liste des colonnes à mettre dans le JSON ---
df_test = df_merged.copy()  # On travaille sur une copie pour éviter de modifier l'original
cols_for_json = [
    'POPULATION_TOTAL', 'SEX_WOMEN', 'SEX_MEN', 'SPC_FARMERS', 'SPC_BUSINESS', 
    'SPC_INACTIVES', 'SPC_EXECUTIVES', 'SPC_EMPLOYEES', 'SPC_WORKERS', 
    'SPC_INTERMEDIATES', 'SPC_RETIRED', 'FAMILY_CHILDREN', 'FAMILY_CHILDLESS', 
    'FAMILY_SINGLE_PARENT', 'ONE_WOMAN', 'ONE_MAN', 'FAMILY', 'HOUSESHARE', 
    'ONE_PERSON', 'AGE_80+', 'AGE_15_24', 'AGE_25_39', 'AGE_40_54', 
    'AGE_55_64', 'AGE_65_79', 'SINGLE', 'DIVORCED', 'LIVEIN_PARTNER', 
    'MARRIED', 'CIVIL_PARTNER', 'WIDOWED', 'Inscrits', 'Abstentions', 
    'Votants', 'Blancs', 'Nuls', 'Exprimés', 'pct_abstention', 
    'pct_blancs', 'pct_nuls'
]

# --- 2. On crée la colonne JSON (au format dictionnaire ou chaîne de caractères) ---

# Option A : Tu veux un vrai dictionnaire Python dans la cellule (recommandé pour la manipulation)
df_test['STATS_JSON'] = df_test[cols_for_json].to_dict(orient='records')

# Option B : Tu veux une vraie chaîne de caractères JSON (le texte '{"POPULATION_TOTAL": ...}')
# df_test['STATS_JSON'] = df_test[cols_for_json].apply(lambda row: row.to_json(), axis=1)


# --- 3. On filtre pour ne garder que ce que tu as demandé ---
final_cols = ['CITY', 'CODE_INSEE', 'pct_gauche', 'pct_centre', 'pct_droite', 'STATS_JSON']
df_final = df_test[final_cols]

In [22]:
df_final.head(10)

,CITY,CODE_INSEE,pct_gauche,pct_centre,pct_droite,STATS_JSON
0,Beauregard,01030,26.93,26.46,46.60,"{'POPULATION_TOTAL': 228.0, 'SEX_WOMEN': 469.0..."
1,Belley,01034,31.86,27.89,40.25,"{'POPULATION_TOTAL': 208.0, 'SEX_WOMEN': 4018...."
2,Bourg-en-Bresse,01053,37.24,30.30,32.45,"{'POPULATION_TOTAL': 240.0, 'SEX_WOMEN': 18622..."
3,Brens,01061,26.09,28.06,45.85,"{'POPULATION_TOTAL': 2367.0, 'SEX_WOMEN': 1520..."
4,Cerdon,01068,21.38,20.43,58.19,"{'POPULATION_TOTAL': 884.996576, 'SEX_WOMEN': ..."
5,Condamine,01112,25.32,23.63,51.05,"{'POPULATION_TOTAL': 246.0, 'SEX_WOMEN': 288.5..."
6,Crans,01129,23.59,25.64,50.77,"{'POPULATION_TOTAL': 86.0, 'SEX_WOMEN': 142.94..."
7,Cuzieu,01141,26.57,34.62,38.81,"{'POPULATION_TOTAL': 1487.001163, 'SEX_WOMEN':..."
8,Montagnieu,01255,23.08,28.99,47.93,"{'POPULATION_TOTAL': 1171.0, 'SEX_WOMEN': 760...."
9,Oyonnax,01283,35.91,26.17,37.91,"{'POPULATION_TOTAL': 48.0, 'SEX_WOMEN': 8952.3..."


In [23]:
df_final.at[0, 'STATS_JSON']

{'POPULATION_TOTAL': 228.0,
 'SEX_WOMEN': 469.00312099999996,
 'SEX_MEN': 400.563806,
 'SPC_FARMERS': 4.67895,
 'SPC_BUSINESS': 49.795161,
 'SPC_INACTIVES': 59.954849,
 'SPC_EXECUTIVES': 119.966302,
 'SPC_EMPLOYEES': 138.692259,
 'SPC_WORKERS': 97.279474,
 'SPC_INTERMEDIATES': 161.4456,
 'SPC_RETIRED': 237.754333,
 'FAMILY_CHILDREN': 59.657917,
 'FAMILY_CHILDLESS': 79.848119,
 'FAMILY_SINGLE_PARENT': 9.981015,
 'ONE_WOMAN': 37.259676,
 'ONE_MAN': 21.291243,
 'FAMILY': 149.487051,
 'HOUSESHARE': 19.96203,
 'ONE_PERSON': 58.550919,
 'AGE_80+': 57.324436000000006,
 'AGE_15_24': 100.022137,
 'AGE_25_39': 191.648144,
 'AGE_40_54': 225.887517,
 'AGE_55_64': 149.881881,
 'AGE_65_79': 160.809887,
 'SINGLE': 206.41612700000002,
 'DIVORCED': 79.80716500000001,
 'LIVEIN_PARTNER': 116.072258,
 'MARRIED': 386.20149299999997,
 'CIVIL_PARTNER': 51.406531,
 'WIDOWED': 45.670426,
 'Inscrits': 568.0,
 'Abstentions': 135.0,
 'Votants': 433.0,
 'Blancs': 2.0,
 'Nuls': 4.0,
 'Exprimés': 427.0,
 'pct_absten

In [27]:
import pandas as pd
import json

# 1. Nettoyage et préparation du JSON
# On s'assure que STATS_JSON est bien traité en chaîne JSON sécurisée pour SQL
df_final['json_str'] = df_final['STATS_JSON'].apply(lambda x: json.dumps(x, ensure_ascii=False).replace("'", "''"))

# 2. Préparation des colonnes textes (Echappement des apostrophes)
df_final['city_escaped'] = df_final['CITY'].astype(str).str.replace("'", "''")

# 3. Construction des requêtes INSERT
table_name = "communes_stats"

df_final['sql_query'] = (
    f"INSERT INTO {table_name} (years, city, code_insee, pct_gauche, pct_centre, pct_droite, statistics) VALUES ("
    "'2022', " # Année en dur
    "'" + df_final['city_escaped'] + "', "
    "'" + df_final['CODE_INSEE'].astype(str) + "', "
    + df_final['pct_gauche'].fillna(0).astype(str) + ", "
    + df_final['pct_centre'].fillna(0).astype(str) + ", "
    + df_final['pct_droite'].fillna(0).astype(str) + ", "
    "'" + df_final['json_str'] + "');"
)

# 4. Exportation dans le fichier SQL avec le schéma complet
with open("insert_communes.sql", "w", encoding="utf-8") as f:
    # --- Schéma de la table ---
    f.write(f"-- 1. Création de la table\n")
    f.write(f"CREATE TABLE IF NOT EXISTS {table_name} (\n")
    f.write("    id SERIAL PRIMARY KEY,\n")
    f.write("    years VARCHAR(4) NOT NULL,\n") # Colonne année
    f.write("    city VARCHAR(255) NOT NULL,\n")
    f.write("    code_insee VARCHAR(10) NOT NULL,\n")
    f.write("    pct_gauche FLOAT DEFAULT 0,\n")
    f.write("    pct_centre FLOAT DEFAULT 0,\n")
    f.write("    pct_droite FLOAT DEFAULT 0,\n")
    f.write("    statistics JSONB,\n")
    f.write("    updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,\n")
    # On rend le couple (année, code_insee) unique pour éviter les doublons
    f.write("    CONSTRAINT unique_city_year UNIQUE (years, code_insee)\n") 
    f.write(");\n\n")

    # --- Index GIN pour les performances JSON ---
    f.write(f"-- 2. Création de l'index GIN pour accélérer les recherches dans le JSON\n")
    f.write(f"CREATE INDEX IF NOT EXISTS idx_{table_name}_stats_gin ON {table_name} USING GIN (statistics);\n\n")
    
    # --- Données ---
    f.write(f"-- 3. Insertions des données\n")
    f.write("\n".join(df_final['sql_query'].tolist()))

print(f"✅ Script SQL généré avec succès dans 'insert_communes.sql' !")

✅ Script SQL généré avec succès dans 'insert_communes.sql' !
